# Outlook

In this notebook we code the Soft Actor-Critic (SAC) algorithm. SAC is an
*off-policy*, maximum-entropy actor-critic method for continuous actions,
described in [this paper](http://proceedings.mlr.press/v80/haarnoja18b/haarnoja18b.pdf)
and [this one](https://arxiv.org/pdf/1812.05905.pdf).

SAC is explained in [this video](https://www.youtube.com/watch?v=U20F-MvThjM)
and [these slides](http://pages.isir.upmc.fr/~sigaud/teach/ps/12_sac.pdf).

It builds on TD3 (twin critics, target networks) and adds an **entropy
bonus**: the agent maximizes reward *and* the entropy of its policy, which
improves exploration. If the entropy coefficient $\alpha$ is set to 0, SAC
reduces exactly to TD3.

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from torch import Tensor
from torch.distributions import Independent, Normal, TanhTransform, TransformedDistribution
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.profiles import Profile
from rl_mind.core import Actor, StochasticAction
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


## The `rl_mind` toolkit

The `rl_mind` package ([documentation](https://master-dac.isir.upmc.fr/rl/rl-mind/)),
installed with the practicals, provides the typed helpers used throughout the
RL notebooks. This one uses
`VecEnv`, `Transitions` / `ReplayBuffer` / `TransitionCollector`, `Evaluator`,
`soft_update`, and the `Actor` base class. Importing `rl_mind.envs` registers
`CartPoleContinuous-v1`.

In [ ]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# SAC

## The Squashed Gaussian actor

SAC uses a **squashed Gaussian** policy: a Gaussian $\mathcal{N}(\mu(s),
\sigma(s))$ passed through a $\tanh$ so that actions stay in $[-1, 1]$.
Since the $\tanh$ is a deterministic, differentiable function, the policy
keeps the property of the Gaussian that matters most here: it can be sampled
with the reparametrization trick (`rsample`), described below.

`TransformedDistribution` handles the change-of-variables correction to the
log-probability automatically. `act` (evaluation) returns the squashed mean.

**Where the log-probabilities come from.** Every loss below needs
$\log \pi(a|s)$ for the action it just sampled. Calling `actor(obs)` returns
a `StochasticAction`, which carries **both**:

- `.value`, the sampled action $a$ (`[B, action_dim]`);
- `.log_prob`, the log-probability $\log \pi(a|s)$ **of that very sample**
  (`[B]`).

The log-probability travels *with* the action rather than being recomputed
later, and this is not just a convenience: the action comes from a
reparametrized sample (`rsample`), so the gradient of the actor loss flows
through it. Recomputing $\log \pi(a|s)$ from a stored action would build a
different graph — and would pay for a second forward pass. This is why you
will read `action.log_prob` and `next_action.log_prob` below instead of
calling some `log_prob(obs, action)` method.

### The reparametrization trick

The actor wants to maximize $\mathbb{E}_{a \sim \pi_\theta(\cdot|s)}\big[
Q(s, a)\big]$ (leaving the entropy aside). The problem: $\theta$ is in the
distribution we sample from, and drawing $a$ at random has no gradient.

The trick is to move the randomness out of $\theta$: draw a noise
$\varepsilon \sim \mathcal{N}(0, I)$, which does not depend on $\theta$, and
compute the action as a differentiable function of it:

$$ a = \tanh\big(\mu_\theta(s) + \sigma_\theta(s)\, \varepsilon\big) $$

The action is now an ordinary output of the network (with some noise added
to the input), so backpropagation goes from $Q(s, a)$ to $a$, then to
$\mu_\theta$ and $\sigma_\theta$: the critic tells the actor *in which
direction to move its actions*, exactly as in DDPG/TD3, but for a stochastic
policy. This requires $Q$ to be differentiable in $a$, which it is (a neural
network).

Without the trick, we would need the score-function estimator of REINFORCE,
$\mathbb{E}\big[Q(s, a)\, \nabla_\theta \log \pi_\theta(a|s)\big]$, which
only uses the *value* of $Q$ and has a much higher variance.

In PyTorch, `dist.rsample()` samples this way (the gradient flows back to
the parameters), while `dist.sample()` does not:

In [ ]:
_mean = torch.zeros(3, requires_grad=True)
_dist = Normal(_mean, torch.ones(3))
print("sample() :", _dist.sample().requires_grad)  # no gradient
print("rsample():", _dist.rsample().requires_grad)  # gradient w.r.t. the mean

In [ ]:
class SquashedGaussianActor(Actor[StochasticAction]):
    def __init__(
        self,
        obs_dim: int,
        hidden: tuple[int, ...],
        action_dim: int,
        min_std: float = 1e-4,
    ):
        super().__init__()
        self.backbone = build_mlp([obs_dim, *hidden], output_activation=nn.ReLU())
        self.mean_layer = nn.Linear(hidden[-1], action_dim)
        self.std_layer = nn.Linear(hidden[-1], action_dim)
        self.softplus = nn.Softplus()
        self.min_std = min_std
        # cache_size=1 avoids numerical issues in the log-prob correction
        self.tanh = TanhTransform(cache_size=1)

    def dist(self, obs: Tensor) -> TransformedDistribution:
        # Build the squashed Gaussian distribution

        features = self.backbone(obs)
        # mean from `self.mean_layer`; std from `self.std_layer`, made positive
        # with `self.softplus`, plus `self.min_std` (keeps it away from 0)
        mean = ...
        std = ...
        # `Normal(mean, std)` is a batch of independent 1D Gaussians:
        # `Independent(..., 1)` makes it one Gaussian over the action vector,
        # so that `log_prob` sums over the action dimensions (shape [B])
        normal = ...
        return TransformedDistribution(normal, [self.tanh])
        assert False, 'Not implemented yet'


    def forward(self, obs: Tensor) -> StochasticAction:
        # Sample an action and compute its log-probability

        dist = self.dist(obs)
        # reparametrized sample (`dist.rsample()`, not `dist.sample()`, so
        # that the gradient flows through the action)
        action = ...
        # log-probability of that very sample (`dist.log_prob`)
        log_prob = ...
        assert False, 'Not implemented yet'

        # Allows the actor to be deep-copied (clears the transform cache)
        self.tanh._cached_x_y = [None, None]
        return StochasticAction(value=action, log_prob=log_prob)

    def act(self, obs: Tensor) -> Tensor:
        features = self.backbone(obs)
        return torch.tanh(self.mean_layer(features))

## The critics $Q(s, a)$

As in TD3, SAC uses two critics (and their target networks) and takes the
minimum of the two to fight the overestimation bias.

In [ ]:
class ContinuousQNetwork(nn.Module):
    """The Q-network $Q(s, a)$ for continuous actions"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int):
        super().__init__()
        self.model = build_mlp([obs_dim + action_dim, *hidden, 1])

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        return self.model(torch.cat([obs, action], dim=1)).squeeze(-1)

## The losses

### Critic loss

The target adds an entropy term to the min of the two target critics,
evaluated at an action sampled from the **current** actor at the next state:

$$ y = r + \gamma\,(1-\text{terminated})\Big(\min_{i=1,2} Q'_{\phi_i}(s',
a') - \alpha \log \pi(a'|s')\Big), \quad a' \sim \pi(\cdot|s') $$

and both critics regress towards it. With $\alpha = 0$ this is exactly the
TD3 critic target.

In [ ]:
def compute_critic_loss(
    gamma: float,
    alpha: float,
    batch: Transitions[StochasticAction],
    actor: SquashedGaussianActor,
    critics: tuple[ContinuousQNetwork, ContinuousQNetwork],
    target_critics: tuple[ContinuousQNetwork, ContinuousQNetwork],
) -> Tensor:
    """Compute the SAC critic loss (sum of the two critics' MSE)"""
    q1, q2 = critics
    tq1, tq2 = target_critics

    # Compute the critic loss

    # action sampled from the CURRENT actor at the next state (no grad)
    with torch.no_grad():
        next_action = actor(batch.next_obs)
        q_next = ...  # min of the two target critics at (next_obs, next_action.value)
        v_next = q_next - alpha * next_action.log_prob
        target = ...
    # the two critics regress towards the shared target
    return ...
    assert False, 'Not implemented yet'


### Actor loss

The actor maximizes the (entropy-augmented) Q-value of its own actions,
$\mathbb{E}_{a \sim \pi(\cdot|s)}\big[\min_{i} Q_{\phi_i}(s, a) - \alpha \log
\pi(a|s)\big]$, i.e. it **minimizes** the opposite:

$$ \mathcal{L}_\pi = \mathbb{E}_{a \sim \pi(\cdot|s)}\big[ \alpha \log
\pi(a|s) - \min_{i=1,2} Q_{\phi_i}(s, a) \big] $$

The action is sampled from the current actor with the reparametrization trick
(so the gradient flows through it). With $\alpha = 0$ this is the TD3 actor
loss.

In [ ]:
def compute_actor_loss(
    alpha: float,
    obs: Tensor,
    actor: SquashedGaussianActor,
    critics: tuple[ContinuousQNetwork, ContinuousQNetwork],
) -> tuple[Tensor, Tensor]:
    """Compute the SAC actor loss; also return the sampled log-probabilities
    (needed to tune the entropy coefficient)"""
    q1, q2 = critics
    # Compute the actor loss

    action = actor(obs)                       # reparametrized sample
    q = ...  # min of the two critics at (obs, action.value)
    actor_loss = ...
    assert False, 'Not implemented yet'

    return actor_loss, action.log_prob

## Hyper-parameters

In [ ]:
@dataclass(frozen=True)
class SACConfig:
    env_name: str = "CartPoleContinuous-v1"
    seed: int = 1

    #: Total number of environment steps
    max_steps: int = Profile.pick(30_000, fast_test=1_200)
    n_envs: int = 8
    steps_per_round: int = 32
    updates_per_round: int = 32
    learning_starts: int = Profile.pick(10_000, fast_test=300)

    buffer_size: int = 1_000_000
    batch_size: int = 256

    gamma: float = 0.99
    tau: float = 0.05
    max_grad_norm: float = 0.5

    #: Entropy coefficient: "auto" tunes it, a float keeps it fixed
    entropy_mode: str = "auto"
    init_entropy_coef: float = 0.2

    actor_hidden: tuple[int, ...] = (256, 256)
    critic_hidden: tuple[int, ...] = (256, 256)
    lr: float = 3e-4

    eval_interval: int = Profile.pick(2_000, fast_test=400)
    n_eval_envs: int = 10

## The entropy coefficient

SAC can **tune** the entropy coefficient $\alpha$ automatically towards a
target entropy $\mathcal{H}_0 = -\dim(\mathcal{A})$. Following SB3, we
optimize $\log \alpha$ (which keeps $\alpha$ positive):

$$ \mathcal{L}_\alpha = -\log\alpha \cdot \big(\log\pi(a|s) +
\mathcal{H}_0\big) $$

In [ ]:
def run_sac(cfg: SACConfig) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    actor = SquashedGaussianActor(env.observation_dim, cfg.actor_hidden, env.action_dim)
    critic_1 = ContinuousQNetwork(
        env.observation_dim, cfg.critic_hidden, env.action_dim
    )
    critic_2 = ContinuousQNetwork(
        env.observation_dim, cfg.critic_hidden, env.action_dim
    )
    target_1 = copy.deepcopy(critic_1)
    target_2 = copy.deepcopy(critic_2)

    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=cfg.lr)
    critic_optimizer = torch.optim.Adam(
        [*critic_1.parameters(), *critic_2.parameters()], lr=cfg.lr
    )

    # Entropy coefficient (tuned or fixed)
    auto_entropy = cfg.entropy_mode == "auto"
    log_alpha = torch.log(torch.ones(1) * cfg.init_entropy_coef).requires_grad_(
        auto_entropy
    )
    alpha_optimizer = torch.optim.Adam([log_alpha], lr=cfg.lr) if auto_entropy else None
    target_entropy = -float(env.action_dim)

    collector = TransitionCollector(env, actor)
    buffer = ReplayBuffer(cfg.buffer_size)
    run_dir = run_directory(f"sac-{cfg.env_name}-S{cfg.seed}")
    writer = SummaryWriter(run_dir)
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=writer,
    )

    pbar = tqdm(total=cfg.max_steps)
    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_round))
        pbar.update(collector.steps - pbar.n)
        if len(buffer) < cfg.learning_starts:
            continue

        for _ in range(cfg.updates_per_round):
            batch = buffer.sample(cfg.batch_size)
            alpha = log_alpha.exp().item()

            # Critic update
            critic_loss = compute_critic_loss(
                cfg.gamma,
                alpha,
                batch,
                actor,
                (critic_1, critic_2),
                (target_1, target_2),
            )
            # Gradient step on the critics

            # zero_grad() on the optimizer, backward() on the loss, then step()
            ...
            assert False, 'Not implemented yet'


            # Actor update
            actor_loss, log_prob = compute_actor_loss(
                alpha, batch.obs, actor, (critic_1, critic_2)
            )
            # Gradient step on the actor

            # same three calls, with the actor's loss and optimizer
            ...
            assert False, 'Not implemented yet'


            # Entropy coefficient update
            if auto_entropy:
                # Entropy coefficient loss, then a gradient step on log_alpha

                # L_alpha (see above), averaged over the batch; `.detach()` the
                # term in log_prob, which must not carry gradients to the actor
                alpha_loss = ...
                # gradient step with `alpha_optimizer`
                ...
                assert False, 'Not implemented yet'


            # Soft update of the two target critics

            # `soft_update(source, target, tau)` moves `target` towards
            # `source` (by a fraction `cfg.tau`), once per critic
            ...
            assert False, 'Not implemented yet'


        writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
        writer.add_scalar("loss/actor", actor_loss.item(), collector.steps)
        writer.add_scalar("entropy_coef", alpha, collector.steps)
        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} best={evaluator.best_reward:7.1f}"
            )

    pbar.close()
    return evaluator

## Launching tensorboard to visualize the results

In [ ]:
setup_tensorboard()

In [ ]:
cfg = SACConfig()
sac = run_sac(cfg)

In [ ]:
sac.visualize_best()

## Learning curve

In [ ]:
plt.plot([r.step for r in sac.history], [r.mean for r in sac.history])
plt.xlabel("steps")
plt.ylabel("cumulated reward")
plt.title(f"SAC on {cfg.env_name}")
plt.show()

## Exercises

- Set `entropy_mode` to a fixed float (e.g. `0.0`) and check that SAC then
  behaves like TD3.
- Run SAC on the harder `Pendulum-v1` environment; you may need to retune the
  hyper-parameters (see the
  [rl-baselines3-zoo](https://github.com/DLR-RM/rl-baselines3-zoo)).